# Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,              
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,              
    "pdf.fonttype": 42,                 
    "ps.fonttype": 42
})
df_deep_train=pd.read_csv("/kaggle/input/multiclass-deep-features/train_features_multiclass.csv")
df_deep_test=pd.read_csv("/kaggle/input/multiclass-deep-features/test_features_multiclass.csv")
df_handcrafted_train=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Hnadcrafted_Train_features_multiclass.csv")
df_handcrafted_test=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Handcraftred_Test_features_multiclass.csv")

In [ ]:
df_deep_test['id']=df_deep_test['image_id']

In [ ]:
df_deep_train['id']=df_deep_train['image_id']

In [ ]:
X_deep_train=df_deep_train.drop(columns=['id','image_id','label'])
X_deep_test=df_deep_test.drop(columns=['id','image_id','label'])

In [ ]:
X_hand_train=df_handcrafted_train.drop(columns=['id','label'])
X_hand_test=df_handcrafted_test.drop(columns=['id','label'])

In [ ]:
y_deep_train=df_deep_train['label']
y_deep_test=df_deep_test['label']
y_hand_train=df_handcrafted_train['label']
y_hand_test=df_handcrafted_test['label']

In [ ]:
X_handcrafted=pd.concat((df_handcrafted_train,df_handcrafted_test),axis=0)

In [ ]:
X_handcrafted.info()

In [ ]:
y_handcrafted=X_handcrafted['label']
X_handcrafted.drop(columns=['id','label'],inplace=True)

In [ ]:
X_deep=pd.concat((df_deep_train,df_deep_test),axis=0)

In [ ]:
X_deep.info()

In [ ]:
y_deep=X_deep['label']
X_deep.drop(columns=['id','label','image_id'],inplace=True)

In [ ]:
X_deep.info()

In [ ]:
X_handcrafted.info()

In [ ]:
df_fusion_train=pd.merge(df_deep_train,df_handcrafted_train,on='id',how='inner')
df_fusion_test=pd.merge(df_deep_test,df_handcrafted_test,on='id',how='inner')

In [ ]:
X_fusion_train=df_fusion_train.drop(columns=['image_id','id','label_y','label_x'])
X_fusion_test=df_fusion_test.drop(columns=['image_id','id','label_y','label_x'])

In [ ]:
y_train=df_fusion_train['label_y']
y_test=df_fusion_test['label_y']

In [ ]:
X_total = np.concatenate((X_fusion_train, X_fusion_test), axis=0)

y_total = np.concatenate((y_train, y_test), axis=0)

In [ ]:
import numpy as np

feature_sets = {
    'Handcrafted': (X_handcrafted, y_handcrafted),
    'Deep': (X_deep, y_deep),
    'Fused': (X_total, y_total),
    'Fused + PCA': (X_total, y_total)
}

print("CLASS DISTRIBUTION ANALYSIS")
print("="*60)

for feat_name, config in feature_sets.items():
    
    if isinstance(config, dict) and 'y' in config:
        y = config['y']
    elif isinstance(config, tuple) or isinstance(config, list):
        y = config[1]
    else:
        print(f"Skipping {feat_name}: Unknown structure")
        continue

    # Safety Check: Ensure y is a valid 1D array
    y = np.array(y) # Convert to numpy array if list/series
    if y.ndim > 1:
        print(f"Warning: {feat_name} y has shape {y.shape}. Flattening...")
        y = y.ravel()

    #Print Distribution
    try:
        unique, counts = np.unique(y, return_counts=True)
        dist_str = ", ".join([f"C{int(c)}={n}" for c, n in zip(unique, counts)])
        print(f"{feat_name:20s}: {dist_str}")
    except Exception as e:
        print(f"{feat_name:20s}: Error calculating distribution - {e}")

print("="*60)

In [ ]:
fix_map = {0: 3, 1: 2, 2: 1, 3: 0}

y_deep = y_deep.map(fix_map)
y_deep_train = y_deep_train.map(fix_map)
y_deep_test = y_deep_test.map(fix_map)

print("Deep labels remapped. Consistency check:")
print("Handcrafted Class 0 count:", np.sum(y_handcrafted == 0))
print("Deep Class 0 count:       ", np.sum(y_deep == 0)) 

# Defining Models and metrics

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, confusion_matrix
)

In [ ]:
#defining classifiers
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models= {
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(n_estimators=50, learning_rate=0.03, random_state=42, eval_metric='logloss'))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", GaussianNB())
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='linear', probability=True, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
}

In [ ]:
#for PCA
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

pca_n_components=0.98

models_PCA = {
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),  
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", XGBClassifier(
            n_estimators=50,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        ))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()), 
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", LogisticRegression(
            max_iter=10000,
            solver='lbfgs',
            penalty='l2',
            C=1.0,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", GaussianNB())
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", SVC(
            kernel='linear', 
            probability=True,
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ]),
}

# MAIN PERFORMANCE TABLE

In [ ]:
def calculate_multiclass_specificity(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)
    specificity_scores = []
    n_classes = len(cm)
    
    for i in range(n_classes):

        tn = np.sum(np.delete(np.delete(cm, i, 0), i, 1))
        fp = np.sum(np.delete(cm[:, i], i))
        
        # Specificity = TN / (TN + FP)
        denominator = tn + fp
        score = tn / denominator if denominator > 0 else 0
        specificity_scores.append(score)
        
    return np.mean(specificity_scores)

In [ ]:
def get_multiclass_auc_scores(y_true, y_proba, classes):
    """
    Calculates Multiclass AUC using One-vs-Rest (OvR) strategy.
    Returns Macro Average and Per-Class scores.
    """
    auc_dict = {}
    
    try:
        auc_avg = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
        auc_dict['AUC_Average'] = auc_avg

        auc_per_class = roc_auc_score(y_true, y_proba, multi_class='ovr', average=None)
        
        for idx, cls in enumerate(classes):
            auc_dict[f'AUC_Class_{cls}'] = auc_per_class[idx]
            
    except ValueError:
        auc_dict['AUC_Average'] = 0.0
        for cls in classes:
            auc_dict[f'AUC_Class_{cls}'] = 0.0

    return auc_dict

In [ ]:
def main_performance_table_multiclass(X, y, models, feature_name="Deep", runs=5, n_splits=5):

    metrics_to_track = [
        'Accuracy', 
        'Precision_Macro', 'Precision_Weighted',
        'Recall_Macro', 'Recall_Weighted',
        'F1_Macro', 'F1_Weighted',
        'Specificity', 'Cohen_Kappa'
    ]
    
    unique_classes = np.unique(y)

    results_store = {
        name: {metric: [] for metric in metrics_to_track} for name in models.keys()
    }
    
    for name in models:
        results_store[name]['AUC_Average'] = []
        for cls in unique_classes:
            results_store[name][f'AUC_Class_{cls}'] = []

    print(f"Starting Multiclass Evaluation: {feature_name} Features | {runs} runs x {n_splits} folds")
    
    for run in range(runs):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        
        for fold_idx, (train_ix, test_ix) in enumerate(skf.split(X, y)):
            X_train_fold, X_test_fold = X[train_ix], X[test_ix]
            y_train_fold, y_test_fold = y[train_ix], y[test_ix]
            
            for name, model in models.items():
                clf = clone(model)
                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('model', clf)
                ])

                pipeline.fit(X_train_fold, y_train_fold)
                y_pred = pipeline.predict(X_test_fold)
                if hasattr(clf, "predict_proba"):
                    y_proba = pipeline.predict_proba(X_test_fold)
                else:
                    y_proba = None
                store = results_store[name]
                
                store['Accuracy'].append(accuracy_score(y_test_fold, y_pred))
                store['Precision_Macro'].append(precision_score(y_test_fold, y_pred, average='macro', zero_division=0))
                store['Precision_Weighted'].append(precision_score(y_test_fold, y_pred, average='weighted', zero_division=0))
                store['Recall_Macro'].append(recall_score(y_test_fold, y_pred, average='macro', zero_division=0))
                store['Recall_Weighted'].append(recall_score(y_test_fold, y_pred, average='weighted', zero_division=0))
                store['F1_Macro'].append(f1_score(y_test_fold, y_pred, average='macro', zero_division=0))
                store['F1_Weighted'].append(f1_score(y_test_fold, y_pred, average='weighted', zero_division=0))
                store['Cohen_Kappa'].append(cohen_kappa_score(y_test_fold, y_pred))

                store['Specificity'].append(calculate_multiclass_specificity(y_test_fold, y_pred))
                
                if y_proba is not None:
                    auc_data = get_multiclass_auc_scores(y_test_fold, y_proba, unique_classes)
                    store['AUC_Average'].append(auc_data['AUC_Average'])
                    for cls in unique_classes:
                        key = f'AUC_Class_{cls}'
                        if key in auc_data:
                            store[key].append(auc_data[key])
                        else:
                            store[key].append(0.0)

    final_rows = []
    
    metric_order = [
        'Accuracy', 
        'Precision_Macro', 'Precision_Weighted', 
        'Recall_Macro', 'Recall_Weighted', 
        'F1_Macro', 'F1_Weighted', 
        'Specificity', 'Cohen_Kappa', 
        'AUC_Average'
    ]
    auc_class_keys = [f'AUC_Class_{c}' for c in unique_classes]
    all_metrics = metric_order + auc_class_keys
    
    for name in models.keys():
        row = {
            "Model": name,
            "Features used": feature_name,
            "Dimensionality reduction": " "
        }
        
        for metric in all_metrics:
            values = results_store[name].get(metric, [])
            if values:
                mean = np.mean(values)
                std = np.std(values)
                row[metric] = f"{mean:.3f} ± {std:.3f}"
            else:
                row[metric] = "N/A"
        
        final_rows.append(row)
    print(final_rows)  
    return pd.DataFrame(final_rows)

In [ ]:
#fusion only
main_performance_table_multiclass(X_total,y_total,models,feature_name='fusion',runs=5,n_splits=5)
#deep only
main_performance_table_multiclass(X_deep.values,y_deep.values,models,feature_name='deep',runs=5,n_splits=5)
#handcrafted feature only
main_performance_table_multiclass(X_handcrafted.values,y_handcrafted.values,models,feature_name='handcrafted',runs=5,n_splits=5)
#pca-optimised features
main_performance_table_multiclass(X_total,y_total,models_PCA,feature_name="Fusion+PCA",runs=5,n_splits=5)

In [ ]:
def specificity_multiclass(y_true, y_pred):
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(cm)):
        tn = np.sum(np.delete(np.delete(cm, i, 0), i, 1))
        fp = np.sum(np.delete(cm[:, i], i))
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return np.mean(specs)

def evaluate_4way_baselines(X_dict, y, n_splits=5, runs=5, pca_param=0.98):
    """
    Evaluates Handcrafted, Deep, Raw Fusion, and Fusion+PCA under identical 5x5 CV.
    StandardScaler and PCA are fitted strictly within each training fold.
    """
    records = []
    unique_classes = np.unique(y)
    
    for feat_name, X in X_dict.items():
        print(f"\n--> Evaluating Feature Space: {feat_name} (Shape: {X.shape})")
        
        # Define pipeline based on feature type
        if feat_name == "Fusion + PCA":
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('pca', PCA(n_components=pca_param, random_state=42)),
                ('clf', LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
            ])
        else:
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
            ])
            
        fold_acc, fold_f1, fold_spec, fold_auc = [], [], [], []
        
        for run in range(runs):
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
            for train_idx, val_idx in skf.split(X, y):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]
                
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_val)
                y_prob = pipe.predict_proba(X_val)
                
                fold_acc.append(accuracy_score(y_val, y_pred))
                fold_f1.append(f1_score(y_val, y_pred, average='macro', zero_division=0))
                fold_spec.append(specificity_multiclass(y_val, y_pred) if len(unique_classes) > 2 
                                  else (np.sum((y_val == 0) & (y_pred == 0)) / np.sum(y_val == 0)))
                
                if len(unique_classes) == 2:
                    fold_auc.append(roc_auc_score(y_val, y_prob[:, 1]))
                else:
                    fold_auc.append(roc_auc_score(y_val, y_prob, multi_class='ovr', average='macro'))
                    
        records.append({
            "Feature Set": feat_name,
            "Accuracy": f"{np.mean(fold_acc)*100:.2f} ± {np.std(fold_acc)*100:.2f}%",
            "Macro-F1": f"{np.mean(fold_f1):.3f} ± {np.std(fold_f1):.3f}",
            "Specificity": f"{np.mean(fold_spec):.3f} ± {np.std(fold_spec):.3f}",
            "Macro AUC": f"{np.mean(fold_auc):.3f} ± {np.std(fold_auc):.3f}"
        })
        
    return pd.DataFrame(records)

In [ ]:

label_mapping = {
    'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3
}

X_multi_dict = {
    "Handcrafted": X_handcrafted.values if hasattr(X_handcrafted, "values") else X_handcrafted,
    "Deep": X_deep.values if hasattr(X_deep, "values") else X_deep,
    "Raw Fusion": X_total,
    "Fusion + PCA": X_total
}

y_multi = np.asarray(y_total, dtype=int)


df_multi_results = evaluate_4way_baselines(
    X_dict=X_multi_dict,
    y=y_multi,
    n_splits=5,
    runs=5,
    pca_param=0.98
)

print("\n--- Multiclass MRI 4-Way Baseline Table ---")
print(df_multi_results.to_markdown(index=False))

# CONFUSION MATRIX for Top Models

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_conf_matrices(y_true, y_pred, title_prefix):
    cm_raw = confusion_matrix(y_true, y_pred)
    cm_norm = confusion_matrix(y_true, y_pred, normalize='true')  # row normalization

    disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_raw)
    disp_raw.plot(cmap='Blues')
    plt.title(f'{title_prefix} - Raw')
    plt.show()

    disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm)
    disp_norm.plot(cmap='Blues')
    plt.title(f'{title_prefix} - Normalized (%)')
    plt.show()

feature_set={
    'Handcrafted': 
    {
        'X' : X_handcrafted,
        'y' : y_handcrafted
    },
    'Deep':
    {
        'X': X_deep,
        'y':y_deep
    },
    'Fusion':
    {
        'X': X_total,
        'y':y_total
    },
    'PCA':
    {
        'X':X_total,
        'y':y_total
    }
}

model_sets = {
    'Handcrafted': 'Random Forest',
    'Deep': 'Logistic Regression',
    'Fusion':'Logistic Regression',
    'PCA': 'Logistic Regression'
}


In [ ]:
from sklearn.model_selection import train_test_split
feature_sets = ['Handcrafted','Deep','Fusion','PCA']
metrics = ['AUC']

# Usage example:
for feat in feature_sets:
    for metric in metrics:
        model_name = model_sets[feat]
        print(f"For {feat} features, top model : {model_name}")
        X=feature_set[feat]['X']
        y= feature_set[feat]['y']
        X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
        print('X_test length : ',len(X_test))
        print('y_test length : ',len(y_test))
        model=models[model_name]
        if feat =='PCA' :
            model=models_PCA[model_name]
        model.fit(X_train,y_train)
        y_pred=model.predict(X_test)
        plot_conf_matrices(y_test, y_pred, title_prefix=f"{feat}-{model_name}")

# COMPUTATION AND EFFICIENCY METRICS

In [ ]:
import time
import sys
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split

In [ ]:
def get_real_param_count(model):
    
    # 1. Unwrap Pipeline if present
    if isinstance(model, Pipeline):
        # The actual classifier is usually the last step
        estimator = model.steps[-1][1]
    else:
        estimator = model

    try:
        # --- Linear Models (Logistic Regression, Linear SVM) ---
        if hasattr(estimator, 'coef_'):
            # (n_classes * n_features) + n_classes (intercepts)
            return estimator.coef_.size + estimator.intercept_.size
        
        # --- Naive Bayes ---
        if hasattr(estimator, 'theta_'): 
            # (n_classes * n_features) means + (n_classes * n_features) variances
            return estimator.theta_.size + estimator.var_.size
            
        # --- Random Forest / Ensembles ---
        if hasattr(estimator, 'estimators_'):
            # Sum of nodes in all decision trees
            return sum([tree.tree_.node_count for tree in estimator.estimators_])
            
        # --- XGBoost ---
        if hasattr(estimator, 'get_booster'):
            booster = estimator.get_booster()
            dump = booster.get_dump()
            # Count split lines to approximate nodes
            return sum([len(tree.split('\n')) for tree in dump])
        
        # --- SVM (Non-linear/RBF) ---
        if hasattr(estimator, 'support_vectors_'):
            # Number of support vectors * n_features
            return estimator.support_vectors_.size
            
    except Exception as e:
        pass
        
    return "N/A"

def compute_metrics(X_train,X_test,y_train,models):
    results_list = []
    
    for model_name, model in models.items():
        print(f"\n=== Measuring: {model_name} ===")

        start_time = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start_time
        print(f"Training Time: {train_time:.4f} s")
        
        inference_times = []
        num_samples = min(20, len(X_test)) 
        
        try: _ = model.predict(X_test[0:1])
        except: pass

        for i in range(num_samples):
            sample = X_test[i:i+1]
            t0 = time.time()
            _ = model.predict(sample)
            t1 = time.time()
            inference_times.append(t1 - t0)
            
        avg_inference = np.mean(inference_times) * 1000 
        std_inference = np.std(inference_times) * 1000 
        print(f"Inference Time: {avg_inference:.4f} ms")
        
        params = get_real_param_count(model)
        try:
            p = pickle.dumps(model)
            memory_mb = len(p) / (1024 * 1024)
        except:
            memory_mb = "N/A"
            
        results_list.append({
            'Model': model_name,
            'Train Time (s)': f"{train_time:.4f}",
            'Inference (ms)': f"{avg_inference:.4f} ± {std_inference:.4f}",
            'Model Size (MB)': f"{memory_mb:.4f}" if isinstance(memory_mb, float) else memory_mb,
            'Parameters': params
        })
        
    return pd.DataFrame(results_list)



In [ ]:
# 1. Clean feature names (remove special characters that upset XGBoost)
X_fusion_train.columns = [str(c).replace('[', '').replace(']', '').replace('<', '') for c in X_fusion_train.columns]
X_fusion_test.columns = [str(c).replace('[', '').replace(']', '').replace('<', '') for c in X_fusion_test.columns]

# 2. Ensure data is numeric (sometimes merge creates 'object' types if IDs are messy)
X_fusion_train = X_fusion_train.apply(pd.to_numeric, errors='coerce').fillna(0)
X_fusion_test = X_fusion_test.apply(pd.to_numeric, errors='coerce').fillna(0)

In [ ]:
#handcrafted features only
metric_hand=compute_metrics(X_hand_train,X_hand_test,y_hand_train,models)
print(metric_hand)
#deep features only
metric_deep=compute_metrics(X_deep_train,X_deep_test,y_deep_train,models)
print(metric_deep)
#fused features only
metric_fusion=compute_metrics(X_fusion_train,X_fusion_test,y_train,models)
print(metric_fusion)
#fused+PCA features only
metric_pca=compute_metrics(X_fusion_train,X_fusion_test,y_train,models_PCA)
print(metric_pca)


## REAL DISAGGREGATED INFERENCE LATENCY 

In [ ]:
import os
import glob
import time
import cv2
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from skimage.feature import local_binary_pattern


X_arr = X_total.values if hasattr(X_total, 'values') else X_total
y_arr = y_total.values if hasattr(y_total, 'values') else y_total

scaler = StandardScaler().fit(X_arr)
pca = PCA(n_components=1151, random_state=42).fit(scaler.transform(X_arr))

clf = LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42)
clf.fit(pca.transform(scaler.transform(X_arr)), y_arr)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking real inference on: {device}")

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)
resnet.eval()
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])

deep_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



def extract_true_radiomics(img_gray):
    img_128 = cv2.resize(img_gray, (128, 128))
    
    _, thresh = cv2.threshold(img_128, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    mean_val   = float(np.mean(img_128))
    std_val    = float(np.std(img_128))
    max_val    = float(np.max(img_128))
    min_val    = float(np.min(img_128))
    median_val = float(np.median(img_128))

    lbp = local_binary_pattern(img_128, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(cnt))
        perimeter = float(cv2.arcLength(cnt, True))
        
        hull = cv2.convexHull(cnt)
        hull_area = float(cv2.contourArea(hull))
        solidity = float(area / (hull_area + 1e-7))
        
    
        if len(cnt) >= 5:
            (x, y), (major_axis, minor_axis), angle = cv2.fitEllipse(cnt)
            a = max(major_axis, minor_axis) / 2.0
            b = min(major_axis, minor_axis) / 2.0
            eccentricity = float(np.sqrt(max(0.0, 1.0 - (b**2 / (a**2 + 1e-7)))))
        else:
            eccentricity = 0.0
    else:
        area, perimeter, eccentricity, solidity = 0.0, 0.0, 0.0, 0.0


    radiomics_19 = np.hstack([
        [mean_val, std_val, max_val, min_val, median_val],
        lbp_hist,
        [area, perimeter, eccentricity, solidity]
    ])
    return radiomics_19

image_folder = "/kaggle/input/brain-tumor-mri-dataset/Testing/*/*"
real_image_paths = glob.glob(image_folder)

if len(real_image_paths) == 0:
    real_image_paths = glob.glob("/kaggle/input/**/**/*.jpg", recursive=True)

assert len(real_image_paths) > 0, "No image paths found. Check your dataset directory!"

sample_paths = real_image_paths[:100]
print(f"Sampled {len(sample_paths)} real MRI images for profiling.")

t_deep, t_rad, t_pca, t_clf = [], [], [], []

dummy_tensor = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    for _ in range(5):
        _ = feature_extractor(dummy_tensor)

sample_warmup = cv2.imread(sample_paths[0], cv2.IMREAD_GRAYSCALE)
_ = extract_true_radiomics(sample_warmup) 

with torch.no_grad():
    for img_path in sample_paths:
        raw_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if raw_gray is None:
            continue
        raw_pil = Image.fromarray(cv2.cvtColor(raw_gray, cv2.COLOR_GRAY2RGB))
        
        # Stage 1: ResNet50 Deep Feature Extraction (GPU)
        img_tensor = deep_transform(raw_pil).unsqueeze(0).to(device)
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        f_deep = feature_extractor(img_tensor).squeeze().cpu().numpy().reshape(1, -1)
        if device.type == 'cuda': torch.cuda.synchronize()
        t_deep.append((time.perf_counter() - t0) * 1000.0)
        
        # Stage 2: Handcrafted Radiomics Extraction (CPU)
        t0 = time.perf_counter()
        f_rad = extract_true_radiomics(raw_gray).reshape(1, -1)
        t_rad.append((time.perf_counter() - t0) * 1000.0)
        
        # Concatenate & Scale
        f_fused = np.hstack((f_deep, f_rad))
        f_fused_scaled = scaler.transform(f_fused)
        
        # Stage 3: PCA Subspace Projection (CPU)
        t0 = time.perf_counter()
        f_pca = pca.transform(f_fused_scaled)
        t_pca.append((time.perf_counter() - t0) * 1000.0)
        
        # Stage 4: Logistic Regression Inference (CPU)
        t0 = time.perf_counter()
        pred_prob = clf.predict_proba(f_pca)
        t_clf.append((time.perf_counter() - t0) * 1000.0)

total_times = np.array(t_deep) + np.array(t_rad) + np.array(t_pca) + np.array(t_clf)

df_latency = pd.DataFrame({
    "Stage": [
        "1. ResNet50 Forward Pass (GPU)",
        "2. Otsu Mask & Radiomics (CPU)",
        "3. PCA Projection (CPU)",
        "4. Logistic Regression Clf (CPU)",
        "Total Pipeline Latency"
    ],
    "Mean (ms)": [np.mean(t_deep), np.mean(t_rad), np.mean(t_pca), np.mean(t_clf), np.mean(total_times)],
    "Std (ms)": [np.std(t_deep), np.std(t_rad), np.std(t_pca), np.std(t_clf), np.std(total_times)],
    "Share (%)": [
        np.mean(t_deep) / np.mean(total_times) * 100,
        np.mean(t_rad) / np.mean(total_times) * 100,
        np.mean(t_pca) / np.mean(total_times) * 100,
        np.mean(t_clf) / np.mean(total_times) * 100,
        100.0
    ]
})

print(df_latency.to_markdown(index=False))

# TSNE AND UMAP VISUALISATION

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

feature_sets = {
    'Handcrafted': (X_handcrafted, y_handcrafted),
    'Deep': (X_deep, y_deep),
    'Fused': (X_total, y_total),
    'Fused + PCA': (X_total, y_total)
}

label_map = {0: 'Glioma', 1: 'Meningioma', 2: 'No Tumor', 3: 'Pituitary'}
n_classes = 4
colors = plt.cm.tab10(np.linspace(0, 1, n_classes))

def run_tsne_visualization():
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    print(f"Computing tsne embeddings...")
    
    for i, (name, (X_data, y_data)) in enumerate(feature_sets.items()):
        ax = axes[i]
        
        X_scaled = StandardScaler().fit_transform(X_data)

        if name == 'Fused + PCA':
            X_scaled = PCA(n_components=0.98, random_state=42).fit_transform(X_scaled)
            
    
        reducer = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
            
        emb = reducer.fit_transform(X_scaled)

        for cls_idx in range(n_classes):
            indices = (y_data == cls_idx)
            if np.sum(indices) > 0:
                ax.scatter(emb[indices, 0], emb[indices, 1], 
                           color=colors[cls_idx], 
                           label=label_map[cls_idx] if i == 0 else "", 
                           s=10, alpha=0.6)
        
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        
        if i == 0: 
            ax.legend(loc='best', markerscale=3, title="Class")
            
    plt.suptitle('TSNE Embeddings: Feature Separability', fontsize=16, y=0.95, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
run_tsne_visualization()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap
import umap.umap_
import sklearn.utils.validation
from importlib import reload


# 2. Define a safe wrapper that handles the version mismatch
def safe_check_array(array, **kwargs):
    # Remove the argument that causes the crash in this sklearn version
    if 'ensure_all_finite' in kwargs:
        kwargs.pop('ensure_all_finite')
    # Call the FRESH, original sklearn function
    return sklearn.utils.validation.check_array(array, **kwargs)

# 3. Patch ONLY UMAP's reference to the function (do not touch sklearn globally)
umap.umap_.check_array = safe_check_array
print(" Environment repaired and UMAP patched successfully.")


y = y_total

label_map = {0: 'Glioma', 1: 'Meningioma', 2: 'No Tumor', 3: 'Pituitary'}
classes = np.unique(y)
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))

feature_sets = {
    'Handcrafted': (X_handcrafted, y_handcrafted),
    'Deep': (X_deep, y_deep),
    'Fused': (X_total, y_total),
    'Fused + PCA': (X_total, y_total)
}

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

print("Generating UMAP embeddings (this should work now)...")

for i, (name, (X_data, y_data)) in enumerate(feature_sets.items()):
    ax = axes[i]
    
    try:
        X_scaled = StandardScaler().fit_transform(X_data)
        if name == 'Fused + PCA':
            X_scaled = PCA(n_components=0.98, random_state=42).fit_transform(X_scaled)
            
        reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
        emb = reducer.fit_transform(X_scaled)
        
        for cls_idx in range(4): 
            indices = (y_data == cls_idx)
            if np.sum(indices) > 0:
                ax.scatter(emb[indices, 0], emb[indices, 1], 
                           color=colors[cls_idx], 
                           label=label_map[cls_idx] if i == 0 else "", 
                           s=10, alpha=0.6)
        
        ax.set_title(name, fontsize=12, fontweight='bold')
        print(f"  - {name} Done.")
        
    except Exception as e:
        print(f"Error processing {name}: {e}")
        ax.text(0.5, 0.5, "Visualization Failed", ha='center', va='center')

    ax.set_xticks([])
    ax.set_yticks([])
    
    if i == 0: 
        ax.legend(loc='best', markerscale=3, title="Class")

plt.suptitle('Figure 7. UMAP Embeddings: Feature Separability', fontsize=16, y=0.95, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('umap_separability_final.png', dpi=300, bbox_inches='tight')
plt.show()

# DE-LONG TEST

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.preprocessing import label_binarize



def compute_midrank(x):
    """Computes midranks for the DeLong algorithm."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

def fastDeLong(predictions_sorted_transposed, label_1_count):
    """
    The Fast DeLong test computation.
    Reference: Sun and Xu, IEEE Signal Proc. Letters, 2014
    """
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)

    for r in range(k):
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
        tx[r, :] = compute_midrank(predictions_sorted_transposed[r, :m])
        ty[r, :] = compute_midrank(predictions_sorted_transposed[r, m:])

    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def calc_pvalue(aucs, sigma):
    """Computes p-value for the null hypothesis (AUC_A == AUC_B)."""
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, sigma), l.T))
    return 2 * stats.norm.sf(np.abs(z))

def get_delong_pvalue(y_true_binary, prob_A, prob_B):
    """
    Wrapper to run DeLong test on two sets of probabilities for binary targets.
    """
    order = (-y_true_binary).argsort()
    label_1_count = int(y_true_binary.sum())
    
    preds_A = prob_A[order]
    preds_B = prob_B[order]
    predictions_sorted = np.vstack((preds_A, preds_B))
    
    aucs, covariance = fastDeLong(predictions_sorted, label_1_count)
    p_value = calc_pvalue(aucs, covariance)
    
    return aucs[0], aucs[1], p_value


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

print("--- FIXING ALIGNMENT ---")

num_samples = len(y_total)
indices = np.arange(num_samples)

train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_total
)


y_train = y_total[train_idx]
y_test  = y_total[test_idx]


X_fusion_train = X_total[train_idx]
X_fusion_test  = X_total[test_idx]


X_deep_train   = X_deep.iloc[train_idx] 
X_deep_test    = X_deep.iloc[test_idx]

X_hand_train   = X_handcrafted.iloc[train_idx]
X_hand_test    = X_handcrafted.iloc[test_idx]

print(f"✅ Alignment Fixed.")
print(f"y_train shape: {y_train.shape}")
print(f"X_deep_train shape: {X_deep_train.shape}")
print(f"First 5 labels in y_train: {y_train[:5]}")

In [ ]:
from sklearn.base import clone
import pandas as pd
import numpy as np

print("--- 1. RETRAINING INDEPENDENT MODELS ---")

# Fusion Model
clf_fused = clone(models['Logistic Regression'])
clf_fused.fit(X_fusion_train, y_train)
print(f"Fusion Model Trained (Expecting {clf_fused.named_steps['scaler'].n_features_in_} features).")

# Deep Model
clf_deep = clone(models['Logistic Regression'])
clf_deep.fit(X_deep_train, y_train)
print(f"Deep Model Trained   (Expecting {clf_deep.named_steps['scaler'].n_features_in_} features).")

# Handcrafted Model 
clf_hand = clone(models['Random Forest'])
clf_hand.fit(X_hand_train, y_train)
print("Handcrafted Model Trained.")

# Fusion + PCA Model
if 'Logistic Regression' in models_PCA:
    clf_PCA = clone(models_PCA['Logistic Regression'])
    clf_PCA.fit(X_fusion_train, y_train)
    print("Fusion+PCA Model Trained.")
else:
    print("Skipping PCA model (not found in dictionary).")

print("\n--- 2. GENERATING PROBABILITIES ---")
# Now each model will correctly accept its specific input shape
prob_fusion = clf_fused.predict_proba(X_fusion_test)
prob_deep   = clf_deep.predict_proba(X_deep_test)
prob_hand   = clf_hand.predict_proba(X_hand_test)
prob_pca    = clf_PCA.predict_proba(X_fusion_test) if 'Logistic Regression' in models_PCA else None

print("Probabilities generated successfully.")

print("\n--- 3. RUNNING DELONG'S TEST ---")

classes = [0, 1, 2, 3]
class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

print(f"{'Class':<12} | {'Comparison':<25} | {'AUC (A)':<8} | {'AUC (B)':<8} | {'p-value':<10} | {'Result'}")
print("-" * 90)

for class_id, class_name in zip(classes, class_names):
    y_test_binary = (y_test == class_id).astype(int)
    

    p_fus  = prob_fusion[:, class_id]
    p_deep = prob_deep[:, class_id]
    p_hand = prob_hand[:, class_id]
    p_pca  = prob_pca[:, class_id] if prob_pca is not None else None

    #  Fusion_PCA vs. Deep
    auc_pca, auc_deep, p_val = get_delong_pvalue(y_test_binary, p_pca, p_deep)
    sig = "**Significant**" if p_val[0][0] < 0.05 else "Not Sig"
    print(f"{class_name:<12} | {'Fusion+PCA vs Deep':<25} | {auc_pca:.4f}   | {auc_deep:.4f}   | {p_val[0][0]:.4f}     | {sig}")

    # Fusion vs. Handcrafted 
    auc_pca, auc_hand, p_val = get_delong_pvalue(y_test_binary, p_pca, p_hand)
    sig = "**Significant**" if p_val[0][0] < 0.05 else "Not Sig"
    print(f"{'':<12} | {'Fusion+PCA vs Handcrafted':<25} | {auc_pca:.4f}   | {auc_hand:.4f}   | {p_val[0][0]:.4f}     | {sig}")
    
    # Fusion vs. Fusion+PCA
    if p_pca is not None:
        auc_fus, auc_pca_val, p_val = get_delong_pvalue(y_test_binary, p_fus, p_pca)
        sig = "**Significant**" if p_val[0][0] < 0.05 else "Not Sig"
        print(f"{'':<12} | {'Fusion+PCA vs Fusion':<25} | {auc_pca_val:.4f}   | {auc_fus:.4f}   | {p_val[0][0]:.4f}     | {sig}")
    
    print("-" * 90)

## NADEAU-BENGIO TEST

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

def nadeau_bengio_corrected_ttest(scores_A, scores_B, n_train_ratio=0.8, n_test_ratio=0.2):
    scores_A = np.asarray(scores_A)
    scores_B = np.asarray(scores_B)
    d = scores_A - scores_B
    k_evals = len(d)  # Total evaluations (25 for 5x5 CV)
    
    mean_diff = np.mean(d)
    var_diff = np.var(d, ddof=1)
    
    # Nadeau and Bengio variance correction factor: (1/K + n_test/n_train)
    corrected_var = ((1.0 / k_evals) + (n_test_ratio / n_train_ratio)) * var_diff
    
    if corrected_var <= 0:
        return 0.0, 1.0, mean_diff
        
    t_stat = mean_diff / np.sqrt(corrected_var)
    df = k_evals - 1
    p_val = 2.0 * stats.t.sf(np.abs(t_stat), df)
    return t_stat, p_val, mean_diff

def collect_fold_accuracies_and_test(X_dict, y, pca_param, n_splits=5, runs=5):
    """
    Runs 5x5 CV across all feature sets in X_dict, collects per-fold accuracies,
    and runs Nadeau-Bengio tests comparing Fusion+PCA against all baselines.
    """
    fold_scores = {name: [] for name in X_dict.keys()}
    
    for run in range(runs):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        for train_idx, val_idx in skf.split(list(X_dict.values())[0], y):
            for name, X in X_dict.items():
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]
                
                if name == "Fusion + PCA":
                    pipe = Pipeline([
                        ('scaler', StandardScaler()),
                        ('pca', PCA(n_components=pca_param, random_state=42)),
                        ('clf', LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
                    ])
                else:
                    pipe = Pipeline([
                        ('scaler', StandardScaler()),
                        ('clf', LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
                    ])
                    
                pipe.fit(X_train, y_train)
                fold_scores[name].append(accuracy_score(y_val, pipe.predict(X_val)))
                
    # Run pairwise Nadeau-Bengio tests
    target = "Fusion + PCA"
    print(f"\n================ NADEAU-BENGIO TEST ({target} vs. Baselines) ================")
    summary_rows = []
    for base_name in X_dict.keys():
        if base_name == target:
            continue
        t_stat, p_val, diff = nadeau_bengio_corrected_ttest(
            scores_A=fold_scores[target], 
            scores_B=fold_scores[base_name],
            n_train_ratio=0.8,
            n_test_ratio=0.2
        )
        sig = "Significant" if p_val < 0.05 else "Not Significant"
        print(f"{target} vs {base_name:<16}: Mean Diff = {diff:+.4f} | t = {t_stat:+.4f} | p-value = {p_val:.4f} ({sig})")
        summary_rows.append({
            "Comparison": f"{target} vs {base_name}",
            "Mean Diff": f"{diff:+.4f}",
            "t-stat": f"{t_stat:.4f}",
            "p-value": f"{p_val:.4f}" if p_val >= 0.001 else "< 0.001",
            "Result": sig
        })
        
    return fold_scores, pd.DataFrame(summary_rows)

In [ ]:
X_multi_dict = {
    "Handcrafted": X_handcrafted.values if hasattr(X_handcrafted, "values") else X_handcrafted,
    "Deep": X_deep.values if hasattr(X_deep, "values") else X_deep,
    "Raw Fusion": X_total,
    "Fusion + PCA": X_total
}
y_multi = np.asarray(y_total, dtype=int)

# Pass pca_param=0.98 for Multiclass
multi_folds, df_multi_tests = collect_fold_accuracies_and_test(
    X_dict=X_multi_dict,
    y=y_multi,
    pca_param=0.98,
    n_splits=5,
    runs=5
)